Parameter Cell

In [6]:
batch_id = ""     # same as 01_landing

StatementMeta(, 2b4b740a-554e-40ff-931f-82ddaad95a95, 8, Finished, Available, Finished, False)

Setup

In [7]:
import sys
sys.path.insert(0, "/lakehouse/default/Files/code")

from bikeshare import control
from bikeshare.audit import RunLogger
from bikeshare.config import make_config

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
cfg = make_config(batch_id=batch_id or None)
control.ensure_control_plane(spark)
log = RunLogger(spark, cfg)
print("batch:", cfg.batch_id, "| env:", cfg.env)

StatementMeta(, 2b4b740a-554e-40ff-931f-82ddaad95a95, 9, Finished, Available, Finished, False)

batch: b003 | env: fabric


Run

In [8]:
from bikeshare import bronze
bronze.run(spark, cfg, log)

StatementMeta(, 2b4b740a-554e-40ff-931f-82ddaad95a95, 10, Finished, Available, Finished, False)

  [bronze ] trips                        SUCCESS  read=15363   written=15363   rejected=0      28.4s
  [bronze ] stations                     SUCCESS  read=2       written=2       rejected=0      11.3s
  [bronze ] weather                      SUCCESS  read=1       written=1       rejected=0      10.2s
  [bronze ] events                       SUCCESS  read=0       written=0       rejected=0       3.2s


Checks

In [9]:
for t in ["bronze.trips", "bronze.stations", "bronze.weather", "bronze.events"]:
    print(t, spark.table(t).count())

spark.table("bronze.trips").select("trip_id", "started_at", "_batch_id", "_source_system", "_source_file", "_ingested_at").show(5, truncate=False)
spark.table("audit.watermarks").show(truncate=False)
spark.table("audit.pipeline_run_log").show(truncate=False)

StatementMeta(, 2b4b740a-554e-40ff-931f-82ddaad95a95, 11, Finished, Available, Finished, False)

bronze.trips 19031
bronze.stations 42
bronze.weather 3
bronze.events 20
+------------+-------------------+---------+--------------+----------------------------------+--------------------------+
|trip_id     |started_at         |_batch_id|_source_system|_source_file                      |_ingested_at              |
+------------+-------------------+---------+--------------+----------------------------------+--------------------------+
|T26080600001|2026-08-06 00:07:01|b003     |trips_sftp    |landing/trips/trips_2026-08-06.csv|2026-11-12 12:57:48.622201|
|T26080600002|2026-08-06 00:30:23|b003     |trips_sftp    |landing/trips/trips_2026-08-06.csv|2026-11-12 12:57:48.622201|
|T26080600003|2026-08-06 00:22:20|b003     |trips_sftp    |landing/trips/trips_2026-08-06.csv|2026-11-12 12:57:48.622201|
|T26080600004|2026-08-06 01:21:28|b003     |trips_sftp    |landing/trips/trips_2026-08-06.csv|2026-11-12 12:57:48.622201|
|T26080600005|2026-08-06 01:20:08|b003     |trips_sftp    |landing/trips/t